In [2]:
# Some import lines
import pandas as pd
import geopandas as gpd
import folium

In [3]:
# Read in the data
# Drop stuff you don't need
# Correct names (you probably won't know what yet, but you'll come back and do this)
# Calculate the new variable(s) you need
pop_df = pd.read_csv('Canada Population 2021 Data.csv')
pop_df = pop_df[pop_df['Location'] != 'Canada'].copy()
pop_df['Total'] = pop_df['First_Nations'] + pop_df['Inuit'] + pop_df['Métis'] + pop_df['Non_Indigenous']
pop_df['pct_indigenous'] = ((pop_df['First_Nations'] + pop_df['Inuit'] + pop_df['Métis']) / pop_df['Total']) * 100
pop_df['pct_str'] = pop_df['pct_indigenous'].round(0).astype(int).astype(str) + '%'

In [4]:
# Here is where you read in the geojson as a df and merge it with your data frame
# You may need to go back and correct names (do this for the data, not the json) and do this again
gdf = gpd.read_file('Canada Provinces Data.json')
merged = gdf.merge(pop_df, left_on='name', right_on='Location', how='left')
merged = merged[~merged['geometry'].isna()]

for col in merged.columns:
    if "datetime" in str(merged[col].dtype).lower():
        merged[col] = merged[col].astype(str)


In [5]:
# Make the map
canada_center = [62.24, -96.28]
m = folium.Map(location=canada_center, zoom_start=3, tiles=None)
# Create 4 feature groups and add each to the map, you might call them, fg1, fg2, fg3, fg4
# The names should be the names you want to show up in your layercontrol
# Here's a skeleton for the first one
fg1 = folium.FeatureGroup(name= 'Percent Indigenous',overlay=False).add_to(m)
fg2 = folium.FeatureGroup(name='First Nations', overlay=False).add_to(m)
fg3 = folium.FeatureGroup(name='Inuit', overlay=False).add_to(m)
fg4 = folium.FeatureGroup(name='Métis', overlay=False).add_to(m)

# Fill in the others
geo_data =  'Canada Provinces Data.json'   #the json file

# Make 4 choropleths, one for each feature group, add each one to the appropriate feature group
# Here is a model 
folium.Choropleth(geo_data=geo_data, data=merged, columns=['Location', 'pct_indigenous'], 
                  key_on='feature.properties.name', fill_color='YlGn', bins=8).geojson.add_to(fg1)

folium.Choropleth(geo_data=geo_data, data=merged, columns=['Location', 'First_Nations'], 
                  key_on='feature.properties.name', fill_color='OrRd', bins=8).geojson.add_to(fg2)

folium.Choropleth(geo_data=geo_data, data=merged, columns=['Location', 'Inuit'], 
                  key_on='feature.properties.name', fill_color='PuBu', bins=8).geojson.add_to(fg3)

folium.Choropleth(geo_data=geo_data, data=merged, columns=['Location', 'Métis'], 
                  key_on='feature.properties.name', fill_color='BuPu', bins=8).geojson.add_to(fg4)
 

#folium.Choropleth(geo_data, name_of_your_merged_df, columns = ['appropriate col 1', 'appropriate col 2'],
#                 key_on='whatever_goes_here', bins = a_number).geojson.add_to(fg1)
# Do this for the other 3, if you are using the same column names for appropriate col 2 every time, you are
# doing it wrong, if you are NOT using the same column name for appropriate col 1, you are doing it wrong

# Use the loop set up below to add the fancy tooltip by adding a geojson to each feature group
# - it will be the SAME GeoJson each time b/c we want the same tooltip every time
for fg in [fg1, fg2, fg3, fg4]:
    folium.features.GeoJson(
        data=merged,
        style_function=lambda x: {'color':'black','fillColor':'transparent','weight':0.5},
        tooltip=folium.features.GeoJsonTooltip(
            fields=['Location', 'pct_str', 'Total', 'First_Nations', 'Inuit', 'Métis'],
            aliases=['Province/Territory:', 'Percent Indigenous:', 'Total Population:', 
                     'First Nations:', 'Inuit:', 'Métis:'],
            localize=True,
            style='background-color:#F0EFEF; border:2px solid black; border-radius:3px; box-shadow:3px; font-size:medium;'
        ),
        highlight_function=lambda x: {'weight':3,'fillColor':'grey'}
    ).add_to(fg)  

# Add a layer control
# Add a title if you want
# Make sure to save it
# Put the name of the map here so you can see it in the notebook
folium.LayerControl(collapsed=False).add_to(m)

title_html = '<h3 align="center" style="font-size:20px;color:black"><b>Indigenous Populations Distribution in Canada (2021)</b></h3>'
m.get_root().html.add_child(folium.Element(title_html))

m.save('map.html')
print("Map saved as 'map.html'")

Map saved as 'map.html'
